In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\santi\OneDrive\Escritorio\proyectos\datos_limpios.csv",
    low_memory=False
)

# 1. Definir pozos eficientes vs ineficientes
# Por qué: necesitamos una métrica clara para clasificarlos
# Usamos producción por hora de flujo efectivo (tef) como medida de productividad
print("=== CLASIFICACIÓN DE POZOS ===")
por_pozo = df.groupby(['idpozo', 'empresa', 'provincia', 'cuenca', 
                        'tipo_de_recurso', 'tipoextraccion', 'formacion'])[
    ['prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'tef']
].sum().reset_index()

# Filtrar solo pozos que produjeron petróleo
por_pozo = por_pozo[por_pozo['prod_pet'] > 0].copy()

# Productividad = producción por hora efectiva de flujo
por_pozo['productividad'] = por_pozo.apply(
    lambda x: x['prod_pet'] / x['tef'] if x['tef'] > 0 else 0, axis=1
)

# Clasificar en terciles: alto, medio, bajo rendimiento
por_pozo['rendimiento'] = pd.qcut(
    por_pozo['productividad'], 
    q=3, 
    labels=['Bajo', 'Medio', 'Alto']
)

print(por_pozo['rendimiento'].value_counts())

# 2. Patrones en pozos de alto rendimiento
# Por qué: identificar qué características comparten los mejores pozos
print("\n=== PATRONES EN POZOS DE ALTO RENDIMIENTO ===")
alto_rendimiento = por_pozo[por_pozo['rendimiento'] == 'Alto']

print("\nDistribución por tipo de recurso:")
print(alto_rendimiento['tipo_de_recurso'].value_counts())

print("\nDistribución por provincia:")
print(alto_rendimiento['provincia'].value_counts())

print("\nDistribución por cuenca:")
print(alto_rendimiento['cuenca'].value_counts())

print("\nDistribución por tipo de extracción:")
print(alto_rendimiento['tipoextraccion'].value_counts())

print("\nTop 10 formaciones más productivas:")
print(alto_rendimiento['formacion'].value_counts().head(10))

# 3. Patrones en pozos de bajo rendimiento
# Por qué: identificar candidatos a cierre o intervención
print("\n=== PATRONES EN POZOS DE BAJO RENDIMIENTO ===")
bajo_rendimiento = por_pozo[por_pozo['rendimiento'] == 'Bajo']

print("\nDistribución por tipo de recurso:")
print(bajo_rendimiento['tipo_de_recurso'].value_counts())

print("\nDistribución por provincia:")
print(bajo_rendimiento['provincia'].value_counts())

print("\nDistribución por cuenca:")
print(bajo_rendimiento['cuenca'].value_counts())

print("\nDistribución por tipo de extracción:")
print(bajo_rendimiento['tipoextraccion'].value_counts())

# 4. Comparación directa alto vs bajo rendimiento
# Por qué: para cuantificar la diferencia y hacer recomendaciones concretas
print("\n=== COMPARACIÓN ALTO VS BAJO RENDIMIENTO ===")
comparacion = por_pozo.groupby('rendimiento').agg(
    cantidad_pozos=('idpozo', 'count'),
    prod_pet_promedio=('prod_pet', 'mean'),
    productividad_promedio=('productividad', 'mean'),
    iny_agua_promedio=('iny_agua', 'mean')
).round(2)
print(comparacion)

# 5. Guardar para Power BI
por_pozo.to_csv(
    r"C:\Users\santi\OneDrive\Escritorio\proyectos\patrones_pozos.csv",
    index=False
)
print("\nArchivo guardado correctamente")

=== CLASIFICACIÓN DE POZOS ===
rendimiento
Alto     9584
Bajo     9583
Medio    9583
Name: count, dtype: int64

=== PATRONES EN POZOS DE ALTO RENDIMIENTO ===

Distribución por tipo de recurso:
tipo_de_recurso
CONVENCIONAL       7628
NO CONVENCIONAL    1955
NO DISCRIMINADO       1
Name: count, dtype: int64

Distribución por provincia:
provincia
Chubut              3417
Neuquén             2525
Mendoza             1394
Santa Cruz          1350
Rio Negro            453
La Pampa             299
Tierra del Fuego      63
Salta                 45
Estado Nacional       20
Formosa               13
Jujuy                  5
Name: count, dtype: int64

Distribución por cuenca:
cuenca
GOLFO SAN JORGE    4689
NEUQUINA           4171
CUYANA              500
AUSTRAL             161
NOROESTE             63
Name: count, dtype: int64

Distribución por tipo de extracción:
tipoextraccion
Bombeo Mecánico              3541
Electrosumergible            2292
Cavidad Progresiva           1767
Surgencia Natural  

C:\Users\santi\AppData\Local\Temp\ipykernel_13320\2810690091.py:74: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  comparacion = por_pozo.groupby('rendimiento').agg(



Archivo guardado correctamente
